In [8]:
import sys
print(sys.executable)

import httpx
print(httpx.__version__)

c:\Users\Admin\Desktop\OpenAI-Day-2\llm_env\Scripts\python.exe
0.27.0


In [9]:
import numpy as np
import os
from dotenv import load_dotenv
from openai import OpenAI
import cohere

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
cohere_api_key = os.getenv("COHERE_API_KEY")

client = OpenAI(api_key=openai_api_key)
co = cohere.Client(cohere_api_key)

In [10]:
documents = [
    # 🔴 Diabetes / medical (core task)
    "The patient has diabetes",
    "The patient does not have diabetes",
    "The patient shows no signs of diabetes",
    "The patient is diabetic",
    "No indication of diabetes was found",
    "Diabetes diagnosis was confirmed",
    "The patient is free from diabetes",
    "The patient might have diabetes",
    "The patient likely does not have diabetes",
    "The patient has uncontrolled diabetes",

    # 🔴 Hypertension / heart (negation traps)
    "The patient suffers from hypertension",
    "The patient does not suffer from hypertension",
    "No evidence of hypertension was observed",
    "Hypertension is present",
    "Hypertension cannot be ruled out",
    "No signs of heart disease",
    "Evidence of heart disease present",
    "Heart disease was not detected",
    "The patient has a history of heart disease",
    "The patient denies heart disease",

    # 🔵 Fever / infection (similar patterns)
    "The patient has a fever",
    "The patient does not have a fever",
    "No fever was recorded",
    "Fever symptoms are present",
    "There is no infection detected",
    "Infection is confirmed",
    "Signs of infection are absent",
    "Possible infection detected",
    "Infection cannot be ruled out",
    "The patient recovered from infection",

    # 🟡 System security (keyword overlap)
    "The system is secure",
    "The system is not secure",
    "The system has vulnerabilities",
    "The system is highly vulnerable",
    "No vulnerabilities were found in the system",
    "The system might be vulnerable",
    "The system is completely safe",
    "Security risks are present in the system",
    "The system is partially secure",
    "The system shows no security issues",

    # 🟢 Ambiguity (bank)
    "The bank approved the loan",
    "The river bank was flooded",
    "He deposited money in the bank",
    "She sat near the river bank",
    "The bank denied the loan",
    "The river overflowed near the bank",
    "He works at a financial bank",
    "Children were playing on the river bank",

    # ⚪ General distractors
    "The car is fast",
    "The car is not fast",
    "The engine is powerful",
    "The engine failed",
    "The weather is hot",
    "The weather is not hot",
    "The device is functioning properly",
    "The device is not working",
]

In [11]:
def get_embeddings_batch(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [np.array(e.embedding) for e in response.data]

In [12]:
doc_embeddings = get_embeddings_batch(documents)

In [13]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [ ]:
query = "The patient is not diabetic but previously had diabetes"
query_vec = get_embeddings_batch([query])[0]

Dense Retrieval (Bi-Encoder)

In [15]:
scores = []

for i, doc_vec in enumerate(doc_embeddings):
    score = cosine_similarity(query_vec, doc_vec)
    scores.append((documents[i], score))

scores.sort(key=lambda x: x[1], reverse=True)

print("Top 10 (Dense Retrieval):\n")
for doc, score in scores[:10]:
    print(f"{score:.4f} -> {doc}")

Top 10 (Dense Retrieval):

1.0000 -> The patient does not have diabetes
0.9102 -> The patient likely does not have diabetes
0.8194 -> The patient shows no signs of diabetes
0.8144 -> The patient has diabetes
0.8121 -> The patient is free from diabetes
0.8002 -> The patient is diabetic
0.7678 -> The patient has uncontrolled diabetes
0.7568 -> The patient might have diabetes
0.7086 -> The patient does not suffer from hypertension
0.6813 -> The patient does not have a fever


In [16]:
for doc, score in scores[:10]:
    if "diabetes" in doc:
        print(doc, score)

The patient does not have diabetes 1.0
The patient likely does not have diabetes 0.9101774223658138
The patient shows no signs of diabetes 0.8193739559511598
The patient has diabetes 0.8144064857648478
The patient is free from diabetes 0.812111835152023
The patient has uncontrolled diabetes 0.7677627046026074
The patient might have diabetes 0.7568014317307926


Cross-Encoder Reranking (Cohere)

In [18]:
top_k_docs = [doc for doc, _ in scores[:10]]

rerank_response = co.rerank(
    model="rerank-english-v3.0",
    query=query,
    documents=top_k_docs
)

reranked = sorted(
    zip(top_k_docs, rerank_response.results),
    key=lambda x: x[1].relevance_score,
    reverse=True
)

print("Top 10 (After Reranking):\n")
for doc, res in reranked:
    print(f"{res.relevance_score:.4f} -> {doc}")

Top 10 (After Reranking):

0.9986 -> The patient does not have diabetes
0.9882 -> The patient likely does not have diabetes
0.9372 -> The patient shows no signs of diabetes
0.8513 -> The patient has diabetes
0.7640 -> The patient is free from diabetes
0.4559 -> The patient is diabetic
0.3616 -> The patient has uncontrolled diabetes
0.0541 -> The patient might have diabetes
0.0009 -> The patient does not suffer from hypertension
0.0003 -> The patient does not have a fever


Compare Before vs After

In [19]:
print("Before (Dense):")
for doc, score in scores[:5]:
    print(doc)

print("\nAfter (Reranked):")
for doc, res in reranked[:5]:
    print(doc)

Before (Dense):
The patient does not have diabetes
The patient likely does not have diabetes
The patient shows no signs of diabetes
The patient has diabetes
The patient is free from diabetes

After (Reranked):
The patient does not have diabetes
The patient likely does not have diabetes
The patient shows no signs of diabetes
The patient has diabetes
The patient is free from diabetes
